# 12 — Final dataset for criteria analysis

Build the final list of works that feed the criteria-extraction pipeline.

Filters applied:

1. `selected_english_translation == 1`
2. Historians: keep only if **all** polities in the work fall in the
   Greek / Greco-Roman groups (non-historians always pass).
3. Drop authors in `EXCLUDE_AUTHORS` (currently `{"new testament"}`).
4. `author_impact_date` must be present.
5. Drop works scored `factuality == 1` (mythic / speculative) from
   `works_factuality_v18.tsv`.
6. Keep only works that fall inside one of the 4 analysis periods
   (last window extended to 230 CE so Diogenes Laertius is included).

**Output**: `data/clean/final/final_dataset_for_criteria.tsv`.

*Follows `notebook_rule.md`: data output is created at the end of the
notebook; every filter step logs its row count.*

## 1. Setup — imports, paths, constants

In [1]:
import json
import random
from pathlib import Path

import pandas as pd

SEED = 0
random.seed(SEED)

META_TSV     = Path('../data/clean/perseus/perseus_works_wikidata.tsv')
FACT_TSV     = Path('../data/clean/final/works_factuality_v18.tsv')
MAPPING_JSON = Path('../data/clean/classifications/works_polity_time_mapping_v2.json')
OUTPUT_TSV   = Path('../data/clean/final/final_dataset_for_criteria.tsv')

EXCLUDE_AUTHORS = {'new testament'}

PERIODS = [
    ('Classical (500–360 BCE)',                        -500, -360),
    ('Late Classical (354–165 BCE)',                   -354, -165),
    ('Hellenistic & Early Roman (165 BCE – 105 CE)',  -165,  105),
    ('High Roman Empire (135–230 CE)',                  135,  230),
]

# Canonical author labels keyed by Perseus author_code. Fixes the
# inconsistent <author> tag in the Perseus TEI XML files.
CANONICAL_AUTHOR = {
    'tlg0062': 'Lucian of Samosata',
    'tlg0061': 'Pseudo-Lucian',
}

# Mirror of POLITY_TO_GROUP from notebook 03. Update both places
# together when a new polity label appears upstream.
POLITY_TO_GROUP = {
    # Greek
    'Classical Greek poleis':                       'Greek',
    'Classical Athens':                             'Greek',
    'Classical Sparta':                             'Greek',
    'Classical Sparta and Peloponnesian League':    'Greek',
    'Classical Thebes':                             'Greek',
    'Classical Greek mercenary polis-world':        'Greek',
    'Archaic Greek poleis':                         'Greek',
    'Archaic Athens (Solonic)':                     'Greek',
    'Archaic Sparta':                               'Greek',
    'Greek philosophical tradition (pan-Hellenic)': 'Greek',
    'Hellenistic Greek philosophical schools':      'Greek',
    'Hellenistic Mediterranean':                    'Greek',
    'Hellenistic Kingdoms':                         'Greek',
    'Hellenistic kingdoms':                         'Greek',
    'Hellenistic Athens':                           'Greek',
    'Hellenistic Sparta':                           'Greek',
    'Hellenistic East':                             'Greek',
    'Macedonian Empire':                            'Greek',
    'Macedonian kingdom':                           'Greek',
    'Macedonian Empire (Successors)':               'Greek',
    'Diadochi Successor States':                    'Greek',
    'Antigonid Dynasty':                            'Greek',
    'Seleucid Empire':                              'Greek',
    'Ptolemaic Kingdom':                            'Greek',
    'Achaean League':                               'Greek',
    'Syracuse':                                     'Greek',
    'Syracuse (Hiero I)':                           'Greek',
    'Sicyon':                                       'Greek',
    'Epirus':                                       'Greek',
    'Corinth':                                      'Greek',
    'Greece':                                       'Greek',
    'Asia Minor':                                   'Greek',
    'Illyrian kingdoms':                            'Greek',
    'Mythological Athens':                          'Greek',
    'Mythological Greece':                          'Greek',
    'Mythological Greece (Odyssey)':                'Greek',
    # Roman
    'Roman Empire':                                 'Roman',
    'Roman Empire (Year of the Four Emperors)':     'Roman',
    'Roman Empire (Flavian)':                       'Roman',
    'Roman Empire (antiquarian)':                   'Roman',
    'Late Roman Republic':                          'Roman',
    'Middle Roman Republic':                        'Roman',
    'Early Roman Republic':                         'Roman',
    'Roman Republic':                               'Roman',
    'Roman Kingdom':                                'Roman',
    'Early Roman Kingdom':                          'Roman',
    'Rome':                                         'Roman',
    'Rome (antiquarian)':                           'Roman',
    'Italy':                                        'Roman',
    'Italic peoples':                               'Roman',
    'Samnite League':                               'Roman',
    'Volscians':                                    'Roman',
    'Gallic tribes':                                'Roman',
    'Carthage':                                     'Roman',
    'Kingdom of Numidia':                           'Roman',
    'Lusitania':                                    'Roman',
    'Celtiberian tribes':                           'Roman',
    'Pontic kingdom (Mithridates VI)':              'Roman',
    # Greco-Roman
    'Roman Empire (Hellenic culture)':              'Greco-Roman',
    'Roman Empire (Greece)':                        'Greco-Roman',
    'Roman Empire (Greece and Thrace)':             'Greco-Roman',
    'Roman Empire (Hellenic intellectual world)':   'Greco-Roman',
    # Persian
    'Achaemenid Empire':                            'Persian',
    'Achaemenid Empire (Cyrus the Great)':          'Persian',
    'Achaemenid Persia':                            'Persian',
    'Achaidmenid Empire':                           'Persian',
    'Lydia':                                        'Persian',
    # Ancient Near East
    'Pharaonic Egypt':                              'Ancient Near East',
    'Pharaonic Egypt (religious-mythic)':           'Ancient Near East',
    'Ancient Israel / Second Temple Judaism':       'Ancient Near East',
    'Judea':                                        'Ancient Near East',
    'Ancient Near East (mythic)':                   'Ancient Near East',
}

GREEK_GROUPS = {'Greek', 'Greco-Roman'}

## 2. Load data

In [2]:
meta_raw = pd.read_csv(META_TSV, sep='\t')
fact_raw = pd.read_csv(FACT_TSV, sep='\t')[['perseus_id',
                                              'factuality',
                                              'factuality_reason']]

print(f'Meta: {len(meta_raw):,} works from {META_TSV.name}')
print(f'Factuality: {len(fact_raw):,} rows from {FACT_TSV.name}')
meta_raw[['file_id', 'perseus_author', 'perseus_title',
          'author_impact_date', 'historian']].head()

Meta: 1,096 works from perseus_works_wikidata.tsv
Factuality: 391 rows from works_factuality_v18.tsv


,file_id,perseus_author,perseus_title,author_impact_date,historian
0,tlg0003.tlg001.perseus-eng6,Thucydides,History of the Peloponnesian War,-425,1
1,tlg0003.tlg001.1st1K-eng1,Thucydides,The Peloponnesian War,-425,1
2,tlg0003.tlg001.perseus-eng4,Thucydides,The History of the Grecian War,-425,1
3,tlg0003.tlg001.1st1K-ger1,Thucydides,Vier Staatsreden aus Thucydides,-425,1
4,tlg0003.tlg001.perseus-grc2,Thucydides,Ἱστορίαι,-425,1


## 3. Preprocessing — apply the six filters

In [3]:
n0 = len(meta_raw)

# Canonicalise inconsistent <author> strings.
for code, label in CANONICAL_AUTHOR.items():
    meta_raw.loc[meta_raw['author_code'] == code, 'perseus_author'] = label

df = meta_raw[meta_raw['selected_english_translation'] == 1].copy()
df = df[~df['perseus_author'].astype(str).str.strip()
                              .str.lower().isin(EXCLUDE_AUTHORS)]
df['year'] = pd.to_numeric(df['author_impact_date'], errors='coerce')
df = df[df['year'].notna()].copy()
df['year'] = df['year'].astype(int)
df['n_pages'] = pd.to_numeric(df['n_pages'], errors='coerce').fillna(0).astype(int)
n1 = len(df)

# Historians: keep only if every polity mentioned in the work falls
# within the Greek / Greco-Roman groups. Non-historians always pass.
with MAPPING_JSON.open() as f:
    polity_mapping = json.load(f)

unmapped = set()
all_greek = {}
for file_id, v in polity_mapping.items():
    polities = [p.strip() for p in
                (v.get('mentioned_polities_in_work') or '').split(';')
                if p.strip()]
    groups = set()
    for p in polities:
        g = POLITY_TO_GROUP.get(p)
        if g is None:
            unmapped.add(p)
        else:
            groups.add(g)
    all_greek[file_id] = bool(groups) and groups.issubset(GREEK_GROUPS)
assert not unmapped, f'Unmapped polities: {sorted(unmapped)}'

df['greek_only'] = df['file_id'].map(all_greek).fillna(False)
n_hist_before = len(df)
df = df[(df['historian'] != 1) | df['greek_only']].copy()
n_hist_dropped = n_hist_before - len(df)
n2 = len(df)

df = df.merge(fact_raw, on='perseus_id', how='left')
df = df[df['factuality'] != 1].copy()
n3 = len(df)


def assign_period(year):
    for label, lo, hi in PERIODS:
        if lo <= year <= hi:
            return label
    return None


df['period'] = df['year'].apply(assign_period)
df = df[df['period'].notna()].copy()
n4 = len(df)

assert df['file_id'].is_unique
assert df['period'].notna().all()

print(f'Raw:                                               {n0:,}')
print(f'After base filters (NT dropped, year required):    {n1:,}  (dropped {n0-n1})')
print(f'After historian/Greek-polity filter:               {n2:,}  '
      f'(dropped {n_hist_dropped} historian works covering non-Greek polities)')
print(f'After factuality != 1 (mythic):                    {n3:,}  (dropped {n2-n3})')
print(f'After period assignment:                           {n4:,}  (dropped {n3-n4})')
print()
print(df['period'].value_counts())
df.head()

Raw:                                               1,096
After base filters (NT dropped, year required):    554  (dropped 542)
After historian/Greek-polity filter:               432  (dropped 122 historian works covering non-Greek polities)
After factuality != 1 (mythic):                    359  (dropped 73)
After period assignment:                           359  (dropped 0)

period
Classical (500–360 BCE)                         162
Late Classical (354–165 BCE)                     80
High Roman Empire (135–230 CE)                   64
Hellenistic & Early Roman (165 BCE – 105 CE)     53
Name: count, dtype: int64


,file_id,perseus_id,perseus_author,wikidata_work_id,wikidata_work_label,selected_english_translation,historian,main_language,author_code,author_wikidata_id,...,form_of_creative_work,confidence,polity_group,keep_greek_focus,is_scientific,year,greek_only,factuality,factuality_reason,period
0,tlg0003.tlg001.1st1K-eng1,tlg0003.tlg001,Thucydides,Q639493,History of the Peloponnesian War,1,1,English,tlg0003,Q41683,...,NaN,100,Greek,1.0,0,-425,True,NaN,NaN,Classical (500–360 BCE)
1,tlg0004.tlg001.perseus-eng2,tlg0004.tlg001,Diogenes Laertius,Q61632188,Lives of Eminent Philosophers 2.5. Socrates,1,1,English,tlg0004,Q59138,...,NaN,100,Greek,1.0,0,210,True,NaN,NaN,High Roman Empire (135–230 CE)
21,tlg0007.tlg054.perseus-eng2,tlg0007.tlg054,Plutarch,Q12902599,Life of Demosthenes,1,1,English,tlg0007,Q41523,...,NaN,100,Greek,1.0,0,75,True,NaN,NaN,Hellenistic & Early Roman (165 BCE – 105 CE)
22,tlg0007.tlg038.perseus-eng2,tlg0007.tlg038,Plutarch,Q100449983,Comparison of Crassus with Nicias,1,1,English,tlg0007,Q41523,...,NaN,100,Greco-Roman,1.0,0,75,True,NaN,NaN,Hellenistic & Early Roman (165 BCE – 105 CE)
23,tlg0007.tlg007.perseus-eng2,tlg0007.tlg007,Plutarch,Q87208306,Life of Solon,1,1,English,tlg0007,Q41523,...,NaN,100,Greek,1.0,0,75,True,NaN,NaN,Hellenistic & Early Roman (165 BCE – 105 CE)


## 4. Summary — period × author/work/page totals

In [4]:
summary = (df.groupby('period')
             .agg(n_authors=('perseus_author', 'nunique'),
                  n_works=('file_id', 'count'),
                  total_pages=('n_pages', 'sum'),
                  total_words=('n_words', 'sum'))
             .reset_index())
summary.loc['TOTAL'] = [
    'TOTAL',
    df['perseus_author'].nunique(),
    len(df),
    int(df['n_pages'].sum()),
    int(df['n_words'].sum()) if 'n_words' in df.columns else 0,
]
summary

,period,n_authors,n_works,total_pages,total_words
0,Classical (500–360 BCE),11,162,6820,1706483
1,Hellenistic & Early Roman (165 BCE – 105 CE),7,53,2780,695630
2,High Roman Empire (135–230 CE),10,64,5835,1458770
3,Late Classical (354–165 BCE),7,80,5089,1272165
TOTAL,TOTAL,34,359,20524,5133048


## 5. Author breakdown per period

In [5]:
author_breakdown = (df.groupby(['period', 'perseus_author'])
                      .agg(n_works=('file_id', 'count'),
                           total_pages=('n_pages', 'sum'))
                      .reset_index()
                      .sort_values(['period', 'total_pages'],
                                   ascending=[True, False]))
author_breakdown

,period,perseus_author,n_works,total_pages
8,Classical (500–360 BCE),Plato,33,2121
9,Classical (500–360 BCE),Thucydides,1,976
6,Classical (500–360 BCE),Isocrates,30,819
2,Classical (500–360 BCE),Aristophanes,11,713
10,Classical (500–360 BCE),Xenophon,6,698
3,Classical (500–360 BCE),Hippocrates,19,597
7,Classical (500–360 BCE),Lysias,34,349
5,Classical (500–360 BCE),Isaeus,12,208
1,Classical (500–360 BCE),Antiphon,6,135
0,Classical (500–360 BCE),Andocides,4,128


## 6. Save the final dataset

In [6]:
output_cols = [
    'file_id', 'perseus_id', 'perseus_author', 'perseus_title',
    'wikidata_work_id', 'wikidata_work_label',
    'author_wikidata_id', 'author_impact_date', 'year', 'period',
    'genre', 'form_of_creative_work', 'instance_of',
    'factuality', 'factuality_reason',
    'main_language', 'languages', 'editors', 'pub_date',
    'n_characters', 'n_words', 'n_pages', 'file_path',
]
output_cols = [c for c in output_cols if c in df.columns]

out = (df[output_cols]
        .sort_values(['period', 'year', 'perseus_author', 'perseus_title'])
        .reset_index(drop=True))

OUTPUT_TSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_TSV, sep='\t', index=False)

print(f'Saved {len(out)} works → {OUTPUT_TSV}')
out.head()

Saved 359 works → ../data/clean/final/final_dataset_for_criteria.tsv


,file_id,perseus_id,perseus_author,perseus_title,wikidata_work_id,wikidata_work_label,author_wikidata_id,author_impact_date,year,period,...,factuality,factuality_reason,main_language,languages,editors,pub_date,n_characters,n_words,n_pages,file_path
0,tlg0027.tlg004.perseus-eng2,tlg0027.tlg004,Andocides,Against Alcibiades,NaN,NaN,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,29491,5226,21,tlg0027/tlg004/tlg0027.tlg004.perseus-eng2.xml
1,tlg0027.tlg002.perseus-eng2,tlg0027.tlg002,Andocides,On His Return,Q27663867,On His Return,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,16263,2941,12,tlg0027/tlg002/tlg0027.tlg002.perseus-eng2.xml
2,tlg0027.tlg001.perseus-eng2,tlg0027.tlg001,Andocides,On the Mysteries,Q87737021,On the Mysteries,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,106628,18619,74,tlg0027/tlg001/tlg0027.tlg001.perseus-eng2.xml
3,tlg0027.tlg003.perseus-eng2,tlg0027.tlg003,Andocides,On the Peace with Sparta,Q87737023,On the Peace with Sparta,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,30121,5221,21,tlg0027/tlg003/tlg0027.tlg003.perseus-eng2.xml
4,tlg0028.tlg006.perseus-eng2,tlg0028.tlg006,Antiphon,On the Choreutes,Q87738364,On the Choreutes,Q15078686,-500,-500,Classical (500–360 BCE),...,4.0,"Antiphon was a forensic orator, and his works ...",English,grc; lat; eng,Kenneth John Maidment,1941,35112,6061,24,tlg0028/tlg006/tlg0028.tlg006.perseus-eng2.xml
